# LAB07　字符串

**对应课程：** 第 7 讲　字符串（遍历与拼接、查找、拆分与清洗、判断与比较、格式化、DNA 翻译）
**时长：** 60 分钟　　**环境：** Jupyter Lab　　**提交：** `学号_姓名_lab07.ipynb`

## 实验目标
1. 熟练使用索引、切片，能预测常用字符串方法的结果
2. 会用“字符串累加器”（从 `""` 开始拼）处理序列
3. 会用 `split`、`strip`、`lower` 拆分和清洗病历文本，用 try-except 检查录入
4. 会用 f-string 控制对齐和小数位数
5. 把 DNA 翻译成蛋白质，看清一个碱基突变的后果；把碱基编码成神经网络能读的数字

## 时间安排
| 部分 | 内容 | 建议用时 |
|---|---|---|
| 0 | 准备：运行第一个代码单元格 | 3 分钟 |
| 1 | 先预测，再运行 | 10 分钟 |
| 2 | 病历文本：清洗、拆分、检查、格式化 | 20 分钟 |
| 3 | DNA 序列：反向互补、查找、合法性、翻译 | 20 分钟 |
| 4 | 神经网络怎么读 DNA：独热编码 | 7 分钟 |
| 5 | 挑战（选做） | — |
| — | 思考题 | 课后完成 |

## 规则
- 每道题写成**函数**，函数名、参数名必须和题目一致；用 `return` 交还结果。
- 学生版里函数体是 `pass`，**写代码时把 `pass` 删掉**。
- check 结果：✅ 正确，❌ 不对，⬜ 还没做。字符串要**一字不差**（包括空格）才算对。
- **不要用 `input()`**。

## 评分标准
| 项目 | 分值 |
|---|---|
| 第 1 部分：预测表填写完整，预测错的写出原因 | 15 |
| 第 2 部分：4 道题 check 全部 ✅ | 35 |
| 第 3 部分：5 道题 check 全部 ✅ | 30 |
| 第 4 部分：2 道题 check 全部 ✅ | 10 |
| 思考题 | 10 |
| 第 5 部分挑战 | 加分 5 |

In [2]:
# ===== 第 0 步：先运行这个单元格（选中后按 Shift + Enter）=====
import hashlib, math, random, statistics

def _norm(v):
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, (int, float)):
        return f"{float(v):.3f}"
    if isinstance(v, (list, tuple)):
        return "[" + ",".join(_norm(x) for x in v) + "]"
    return str(v)

_ANS = {
    "2.1 normalize_sex": "bd8a689d4c",
    "2.2 parse_record": "d60d833e50",
    "2.3 is_valid_temp": "323acd584c",
    "2.4 report_line": "a7350d78ef",
    "3.1 rev_comp": "fe988f1dde",
    "3.2 count_overlap": "2e96d765b7",
    "3.3 is_dna": "073c63eef5",
    "3.4 translate": "30996bc28d",
    "3.5 first_diff": "6684120ec1",
    "4.1 one_hot": "3caf43d150",
    "4.2 gc_by_neuron": "4647bc9f6f",
    "5.1 translate_from_atg": "d4a61ffd74",
    "5.2 汉字个数": "c66c55ec75",
    "5.2 数字个数": "1ce52611d9",
}

def check(name, value):
    """把你的结果和参考答案对照。它只告诉你对不对，不会把答案告诉你。"""
    if value is None or (isinstance(value, (list, tuple)) and len(value) > 0 and all(v is None for v in value)):
        print(f"⬜ {name}：还没完成")
        return
    code = hashlib.md5(_norm(value).encode("utf-8")).hexdigest()[:10]
    if code == _ANS.get(name):
        print(f"✅ {name}：{value!r}  正确")
    else:
        print(f"❌ {name}：你的结果是 {value!r}，和参考答案不一致，再检查一下")

print("准备完成，可以开始了！")




准备完成，可以开始了！


## 遇到问题先查这里

| 现象 | 常见原因 | 怎么改 |
|---|---|---|
| `TypeError: can only concatenate str (not "int") to str` | 字符串和数字直接用 `+` 拼接 | 用 f-string，或先 `str()` |
| `TypeError: 'str' object does not support item assignment` | 想直接改字符串里的某个字符 | 字符串不可变，拼一个新串 |
| `IndexError: string index out of range` | 下标 ≥ 长度 | 最大下标是 `len(s) - 1` |
| `ValueError: could not convert string to float` | `float("abc")` 这类转换 | 用 try-except 捕获 |
| `ValueError: not enough values to unpack (expected 2, got 1)` | `split` 的结果不是两段（没有冒号，或多了空项） | 先 `print(item.split(":"))` 看看拆出了什么 |
| 结果看起来对，check 却是 ❌ | 多了或少了空格；大小写不一致 | 用 `repr(结果)` 打印，能看清空格；必要时 `strip()` |
| 调用了 `s.strip()`，s 却没变 | 方法返回新串，原串不变 | 写成 `s = s.strip()` |
| find 的结果拿去切片，结果很怪 | 没找到时 find 返回 -1，而 -1 是最后一个字符的下标 | 先判断 `pos != -1` |

## 第 1 部分　先预测，再运行（10 分钟）

### 1.1　索引、切片和方法
`seq = "ATGGCGTACGTTAGCC"`（16 个碱基）。先填“我的预测”，再运行核对。

| 表达式 | 我的预测 | 运行结果 | 为什么（预测错时填写） |
|---|---|---|---|
| `seq[0]`, `seq[-1]` |A C| A C| |
| `seq[3:6]` |GCG |GCG | |
| `seq[:3]`, `seq[-3:]` |ATG GCC |ATG GCC | |
| `seq[::4]` |ACCA |ACCA | |
| `seq.find("TAC")` |6 |6 | |
| `seq.find("AAA")` | -1|-1 | |
| `seq.count("G")` |5 | 5| |
| `seq.lower()[:3] + "C" * 3` |agtgacCCC |atgCCC  |看错了|
| `"10" < "9"` | True| True| |
| `"36.5".isdigit()` | False|False | |

In [2]:
seq = "ATGGCGTACGTTAGCC"
print(seq[0], seq[-1])
print(seq[3:6])
print(seq[:3], seq[-3:])
print(seq[::4])
print(seq.find("TAC"))
print(seq.find("AAA"))
print(seq.count("G"))
print(seq.lower()[:3] + "C" * 3)
print("10" < "9")
print("36.5".isdigit())

A C
GCG
ATG GCC
ACCA
6
-1
5
atgCCC
True
False


### 1.2　字符串累加器
| 代码 | 我的预测 | 运行结果 |
|---|---|---|
| A |CTAG |CTAG |
| B |GGAATTCC |GGAATTCC |

In [3]:
# A
r = ""
for b in "GATC":
    r = b + r
print(r)

CTAG


In [4]:
# B
r = ""
for b in "GATC":
    r = r + b * 2
print(r)


GGAATTCC


## 第 2 部分　病历文本（20 分钟）

### 2.1　统一性别录入
写函数 `normalize_sex(s)`：
- 先去掉首尾空白、转成小写
- `"男"`、`"m"`、`"male"` → 返回 `"男"`
- `"女"`、`"f"`、`"female"` → 返回 `"女"`
- 其他 → 返回 `"未知"`

In [5]:
def normalize_sex(s):
    # ↓↓↓ 在这里写代码（删掉 
    s=s.strip().lower()
    if s in["男","m","male"]:
        return "男"
    elif s in ["女","f","female"]:
        return"女"
    else:
        return "未知"
    # ↑↑↑

raw = [" 男", "M", "male ", "Female", "女 ", "f", "?", "MALE"]
result = [normalize_sex(s) for s in raw]     # 对 raw 里的每一项都调用一次
print(result)
check("2.1 normalize_sex", result)

['男', '男', '男', '女', '女', '女', '未知', '男']
✅ 2.1 normalize_sex：['男', '男', '男', '女', '女', '女', '未知', '男']  正确


### 2.2　从一行病历里取出姓名、年龄、体温
病历的每一项用分号 `;` 隔开，项目名和值用冒号 `:` 隔开，**两边可能有多余的空格**。

写函数 `parse_record(record)`，返回三个值：姓名（字符串）、年龄（int）、体温（float），即 `return name, age, temp`。

提示：`for item in record.split(";"):` → `key, value = item.split(":")` → 两边都要 `strip()` → 用 if 判断 key 是哪一项。

In [15]:
def parse_record(record):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    for item in record.split(";"):
        key,value=item.split(":")
        key=key.strip()
        value=value.strip()
        if key=="姓名":
            name=value
        elif key=="年龄":
            age=int(value)
        elif key=="体温":
            temp=float(value)
        else:
            gender=value        
    return name,age,temp
        
    # ↑↑↑

r1 = "姓名:张三;性别:男;年龄:45;体温:38.6"
r2 = " 姓名: 李四 ; 性别:女; 年龄: 62 ;体温:37.1 "
print(parse_record(r1))
print(parse_record(r2))
check("2.2 parse_record", [parse_record(r1), parse_record(r2)])


('张三', 45, 38.6)
('李四', 62, 37.1)
✅ 2.2 parse_record：[('张三', 45, 38.6), ('李四', 62, 37.1)]  正确


### 2.3　体温录入合法吗？
写函数 `is_valid_temp(s)`：
- 能转成数字（用 **try-except**，不要用 `isdigit`），并且
- 数值在 34～43 之间（含两端）

才返回 `True`，否则返回 `False`。

In [14]:
def is_valid_temp(s):
    
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    try:
        num=float(s)
        if 34<=num<=43:
            return True
        else:return False
    except ValueError:
        return False
    
    # ↑↑↑

tests = ["36.5", "abc", "120", "37", " 38.2 ", "-5", "三十七"]
check("2.3 is_valid_temp", [is_valid_temp(s) for s in tests])

✅ 2.3 is_valid_temp：[True, False, False, True, True, False, False]  正确


### 2.4　打印对齐的体温单
写函数 `report_line(name, temp)`，返回一行字符串（不是 print！），格式为：
- 姓名**左对齐，占 4 格**
- 体温**右对齐，占 6 格，保留 1 位小数**
- 紧接着 `℃`，再一个空格，再写 `发热`（≥ 37.3）或 `正常`

例如 `report_line("张三", 38.6)` 返回 `'张三    38.6℃ 发热'`（“张三”后补 2 个空格，38.6 前补 2 个空格）。

提示：`f"{name:<4}{temp:>6.1f}℃ "` 再拼上 `"发热"` 或 `"正常"`。写好后用循环打印整张体温单。

In [3]:
def report_line(name, temp):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    if temp>=37.3:
        status="发热"
    else:status="正常"
    return f"{name:<4}{temp:>6.1f}℃ {status}"
    # ↑↑↑

names = ["张三", "李四", "欧阳娜娜"]
values = [38.6, 36.9, 37.3]
for i in range(len(names)):
    print(report_line(names[i], values[i]))
check("2.4 report_line", [report_line("张三", 38.6), report_line("李四", 36.9), report_line("欧阳娜娜", 37.3)])

张三    38.6℃ 发热
李四    36.9℃ 正常
欧阳娜娜  37.3℃ 发热
✅ 2.4 report_line：['张三    38.6℃ 发热', '李四    36.9℃ 正常', '欧阳娜娜  37.3℃ 发热']  正确


## 第 3 部分　DNA 序列（20 分钟）

### 3.1　反向互补链
写函数 `rev_comp(seq)`：先用循环逐个碱基求互补链（A↔T，G↔C，从 `""` 开始拼），再反过来，返回反向互补链。

In [6]:
def rev_comp(seq):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    n_seq=""
    for b in seq:
        if b =="A":
            n="T"
        elif b=="G":
            n="C"
        elif b=="C":
            n="G"
        else :
            n="A"
        n_seq+=n
    return n_seq[::-1]
    
    # ↑↑↑

check("3.1 rev_comp", [rev_comp("ATGC"), rev_comp("AAGCTT"), rev_comp("ATGGCGTACGTTAGCC")])

✅ 3.1 rev_comp：['GCAT', 'AAGCTT', 'GGCTAACGTACGCCAT']  正确


### 3.2　找出所有出现位置（可重叠）
`count()` 不数重叠的出现：`"TATATATA".count("TATA")` 是 2，但 TATA 其实出现在下标 0、2、4 三处。

写函数 `count_overlap(seq, motif)`：用 `find` + `while`，从上一次找到的位置**后一位**接着找，返回出现次数。

In [9]:
def count_overlap(seq, motif):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    count,start=0,0
    while True:
        pos=seq.find(motif,start)
        if pos==-1:
            break
        count+=1
        start=pos+1
    return count
    # ↑↑↑

    

print("TATATATA".count("TATA"), count_overlap("TATATATA", "TATA"))
check("3.2 count_overlap", [count_overlap("TATAAAGCTATAAATTATAAA", "TATA"),
                           count_overlap("TATATATA", "TATA"),
                           count_overlap("GGGG", "GG"),
                           count_overlap("ATGC", "TTT")])

2 3
✅ 3.2 count_overlap：[3, 3, 3, 0]  正确


### 3.3　这是一段合法的 DNA 吗？
写函数 `is_dna(seq)`：不区分大小写，只要出现 A、C、G、T 以外的字符就返回 `False`，否则返回 `True`。
注意 `return True` 的位置（回顾 LAB06 的 1.2）。

In [12]:
def is_dna(seq):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    for b in seq:
        if b.upper() not in ["A","G","C","T"]:
            return False
    return True
    
    # ↑↑↑

check("3.3 is_dna", [is_dna("ATGC"), is_dna("atgc"), is_dna("ATGX"), is_dna(""), is_dna("ACGTN")])

✅ 3.3 is_dna：[True, True, False, True, False]  正确


### 3.4　DNA → 蛋白质
先运行下面的单元格（课件第 19 页的查表函数）。然后写函数 `translate(seq)`：
- 每 3 个碱基一组（`range(0, len(seq) - 2, 3)` + 切片），用 `codon_to_aa` 查出氨基酸，拼起来
- 遇到终止密码子（`"*"`）就停止，`*` 本身不放进结果

In [13]:
BASES = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"

def codon_to_aa(codon):
    i = BASES.find(codon[0])
    j = BASES.find(codon[1])
    k = BASES.find(codon[2])
    return AA[16 * i + 4 * j + k]

print(codon_to_aa("ATG"), codon_to_aa("GAG"), codon_to_aa("GTG"), codon_to_aa("TAA"))

M E V *


In [17]:
def translate(seq):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    protein=""
    for i in range(0,len(seq)-2,3):
        coden=seq[i:i+3]
        aa=codon_to_aa(coden)
        if aa=="*":
            break
        protein+=aa
    return protein

    
    # ↑↑↑

normal = "ATGGTGCACCTGACTCCTGAGGAGAAGTCT"    # β-珠蛋白基因开头 30 个碱基
sickle = "ATGGTGCACCTGACTCCTGTGGAGAAGTCT"    # 镰状细胞贫血
print(translate(normal))
print(translate(sickle))
check("3.4 translate", [translate(normal), translate(sickle), translate("ATGGCTGAACGTTGGTAAGCC")])

MVHLTPEEKS
MVHLTPVEKS
✅ 3.4 translate：['MVHLTPEEKS', 'MVHLTPVEKS', 'MAERW']  正确


### 3.5　突变在哪里？
写函数 `first_diff(p1, p2)`：逐个位置比较两个等长字符串，返回**第一个不同字符的下标**；完全相同时返回 -1。
用它找出正常和突变蛋白第一个不同的位置，存入 `pos`。

In [18]:
def first_diff(p1, p2):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    for b in range(len(p1)):
        if p1[b]!=p2[b]:
            return b
    return -1
    pos=b
    

pos = first_diff(translate(normal), translate(sickle))
print(pos)
check("3.5 first_diff", pos)

6
✅ 3.5 first_diff：6  正确


## 第 4 部分　神经网络怎么读 DNA（7 分钟）

### 4.1　独热编码
写函数 `one_hot(b)`：返回一个 4 个数的列表，依次表示“是不是 A、C、G、T”。
提示：`int(b == "A")` 在 b 是 A 时为 1，否则为 0（第三章：布尔值也能当数字）。

In [4]:
def one_hot(b):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    return [int(b=="A"),int(b=="C"),int(b=="G"),int(b=="T")]
    # ↑↑↑


check("4.1 one_hot", [one_hot("A"), one_hot("C"), one_hot("G"), one_hot("T")])

✅ 4.1 one_hot：[[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]  正确


### 4.2　一个“数 GC”的神经元
先运行下面的 `weighted_sum`（LAB06 写过）。再写函数 `gc_by_neuron(seq)`：
- 权重 `w = [0, 1, 1, 0]`（只关心 C 和 G），偏置 0
- 对每个碱基：`one_hot(b)` 和 w 做加权求和，累加起来
- 最后除以序列长度，返回结果

它应该和 GC 含量一模一样。

In [5]:
def weighted_sum(x, w, b):
    z = b
    for i in range(len(x)):
        z = z + x[i] * w[i]
    return z

In [7]:
def gc_by_neuron(seq):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓
    w = [0, 1, 1, 0]
    total=0
    for b in seq:
        total+=weighted_sum(one_hot(b),w,0)
    return total/len(seq)
    # ↑↑↑
    
###xuyaofuxi-silver
seq30 = "ATGGCGTACGTTAGCCGATAAGCTTGCACG"
print(gc_by_neuron(seq30), (seq30.count("G") + seq30.count("C")) / len(seq30))
check("4.2 gc_by_neuron", gc_by_neuron(seq30))

0.5333333333333333 0.5333333333333333
✅ 4.2 gc_by_neuron：0.5333333333333333  正确


## 第 5 部分　挑战（选做）

### 5.1　从第一个 ATG 开始翻译
真实序列前面常有一段非编码区。写函数 `translate_from_atg(seq)`：找到**第一个** `ATG`，从那里开始翻译（复用 `translate`）；没有 ATG 时返回空字符串 `""`。

In [ ]:
def translate_from_atg(seq):
    # ↓↓↓ 在这里写代码（删掉 pass）↓↓↓

    # ↑↑↑

check("5.1 translate_from_atg", [translate_from_atg("CCGATGGCTGAACGTTGGTAAGCC"),
                                 translate_from_atg("ATGGTGCACCTGACTCCTGAG"),
                                 translate_from_atg("CCCGGGTTT")])

### 5.2　病历文本里有几个汉字、几个数字？
`text = "患者男，45岁，体温38.6℃，心率110次/分"`

数出汉字个数 `n_han` 和数字字符个数 `n_digit`。
提示：常用汉字的编码在 `"\u4e00"`（一）到 `"\u9fff"` 之间，可以用 `"\u4e00" <= c <= "\u9fff"` 判断；数字用 `c.isdigit()`。

In [ ]:
text = "患者男，45岁，体温38.6℃，心率110次/分"
n_han = None
n_digit = None
# ↓↓↓ 在这里写代码 ↓↓↓


# ↑↑↑
check("5.2 汉字个数", n_han)
check("5.2 数字个数", n_digit)

## 思考题（课后完成，写在下面的单元格里）
1. 1.2 中 A 和 B 的区别是什么？用 A 的思路，怎样一步写出反向互补链？
2. 2.2 中如果不 `strip()`，哪一条记录会出问题？会报什么错？为什么？
3. 2.3 为什么不能用 `isdigit()`？“格式合法”和“数值合理”有什么区别？
4. 3.2 中，`find(motif, pos + 1)` 如果写成 `find(motif, pos)` 会怎样？
5. 3.5 中，一个碱基的改变造成了什么后果？如果 GAG 变成 GAA（也编码谷氨酸），蛋白质会改变吗？用你的 translate 验证一下。

**我的回答：**

1.b和r的先后顺序不一样 且B多复制了字符串 comp=new+comp

2.格式就不对啊 都不知道从哪里分开

3.isdigit只能识别0到9 两位数他就不认了 格式合法是是不是只有再要求范围内的东西 数值合理是是不是数字有没有实际意义
4.找到一个后不加1的话再开始还会停留在找到的下标会导致重复

5. 取决于密码子的简并性 蛋白质不会改变 殊途同归

## 提交前检查

- [ ] **Kernel → Restart Kernel and Run All Cells**，从头到尾没有报错
- [ ] 所有 `check` 都是 ✅（挑战题可以是 ⬜）
- [ ] 第 1 部分的预测表都填写了，预测错的写了原因
- [ ] 思考题已作答
- [ ] 保留运行结果（不要清除输出），保存后提交：`学号_姓名_lab07.ipynb`